In [1]:
%load_ext jupyter_black

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [3]:
import sys
from pathlib import Path

# Add project root to Python path to enable src imports
project_root = Path.cwd().parent
sys.path.append(str(project_root))

import os
import time
import json
import pickle
import pandas as pd
from dotenv import load_dotenv

import torch
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer, CrossEncoder

from src.retrieval.reranker import Reranker
from src.evaluation.metrics import evaluate_pipeline
from src.retrieval.hybrid_retriever import (
    dense_search,
    bm25_search,
    reciprocal_rank_fusion,
)

In [4]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
device

'mps'

In [5]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [6]:
embed_model = SentenceTransformer(
    "intfloat/multilingual-e5-small", token=hf_token, device=device
)

client = QdrantClient("localhost", port=6333)

with open("../data/processed/bm25_index.pkl", "rb") as f:
    bm25_index = pickle.load(f)

bm25, chunks = bm25_index["bm25"], bm25_index["chunks"]

reranker_model = Reranker()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7143.87it/s]


In [7]:
# Dataset yükle
with open("../data/evaluation/questions.json", encoding="utf-8") as f:
    dataset = json.load(f)

In [8]:
# Pipeline 1: Naive
def naive_pipeline(query: str) -> list[str]:
    results = dense_search(query, embed_model, client, top_k=10)
    return [chunk_id for chunk_id, _ in results]

In [9]:
# Pipeline 2: Hybrid
def hybrid_pipeline(query: str) -> list[str]:
    dense = dense_search(query, embed_model, client, top_k=20)
    sparse = bm25_search(query, bm25, chunks, top_k=20)
    fused = reciprocal_rank_fusion(dense, sparse, chunks, top_k=10)
    return [c["chunk_id"] for c in fused]

In [10]:
# Pipeline 3: Hybrid + Reranker
def hybrid_reranker_pipeline(query: str) -> list[str]:
    dense = dense_search(query, embed_model, client, top_k=20)
    sparse = bm25_search(query, bm25, chunks, top_k=20)
    fused = reciprocal_rank_fusion(dense, sparse, chunks, top_k=20)
    reranked = reranker_model.rerank(query, fused, top_k=10)
    return [c["chunk_id"] for c in reranked]

In [11]:
pipelines = {
    "Naive": naive_pipeline,
    "Hybrid": hybrid_pipeline,
    "Hybrid + Reranker": hybrid_reranker_pipeline,
}

In [12]:
results = []
for name, fn in pipelines.items():
    print(f"Evaluating: {name}...")
    t0 = time.perf_counter()
    scores = evaluate_pipeline(fn, dataset)
    elapsed = time.perf_counter() - t0

    results.append({"Pipeline": name, **scores, "Süre (sn)": round(elapsed, 1)})
    print(f"  MRR: {scores['MRR']} | Hit@5: {scores['Hit@5']}")

Evaluating: Naive...
  MRR: 0.7384 | Hit@5: 0.8889
Evaluating: Hybrid...
  MRR: 0.7872 | Hit@5: 0.9111
Evaluating: Hybrid + Reranker...
  MRR: 0.7272 | Hit@5: 0.9


In [13]:
df = pd.DataFrame(results).set_index("Pipeline")
print("\n=== SONUÇLAR ===")
df


=== SONUÇLAR ===


,MRR,Hit@1,Hit@3,Hit@5,Hit@10,Süre (sn)
Pipeline,,,,,,
Naive,0.7384,0.6333,0.8222,0.8889,0.9333,5.0
Hybrid,0.7872,0.6778,0.9000,0.9111,0.9667,2.4
Hybrid + Reranker,0.7272,0.6222,0.8222,0.9000,0.9444,174.3
